# PDE-Based Image Inpainting: Model Comparison

**Authors:** Blake Taylor, Fang Fang, Inzaghi Moniaga

This notebook demonstrates four PDE-based inpainting methods. All models solve the steady-state PDE **∇·(G∇u) = 0** in Ω, with **u = f** on ∂Ω, differing only in the choice of conductivity G.

| Model | Conductivity G | Key property |
|-------|----------------|--------------|
| **Harmonic** | G = 1 | Isotropic diffusion, blurs edges |
| **TV** | G = 1/\|∇u\| | Edge-preserving |
| **CDD** | G = g(\|κ\|)/\|∇u\| | Curvature-driven, connectivity principle |
| **QCDD** | G = g(\|κ\|) | Faster curvature-driven (no gradient denominator) |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter, binary_dilation, distance_transform_edt
from PIL import Image

print("Libraries loaded.")

## 1. Model Implementations

### 1.1 Harmonic Inpainting

Solves the Laplace equation **Δu = 0** via Gauss-Seidel iteration.
The mask region is initialised from adjacent boundary pixels rather than the global image mean — this preserves local intensity structure at the mask edges.

In [ ]:
def harmonic_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    max_iter: int = 5000,
    tol: float = 1e-5,
    verbose: bool = True,
) -> tuple:
    """Harmonic inpainting via Gauss-Seidel iteration."""
    u = image.copy().astype(np.float64)

    # Initialise from boundary pixels to preserve local structure
    dilated = binary_dilation(mask)
    boundary = dilated & ~mask  # non-mask pixels adjacent to mask

    if np.any(boundary):
        u[mask] = image[boundary].mean()
    elif np.any(~mask):
        u[mask] = image[~mask].mean()
    else:
        u[mask] = 0.5

    residuals = []

    for iteration in range(max_iter):
        u_old = u.copy()

        u_pad = np.pad(u, 1, mode='edge')

        # 4-neighbour average enforces Δu = 0 at steady state
        u_new = 0.25 * (
            u_pad[1:-1, 2:]   +  # East
            u_pad[1:-1, :-2]  +  # West
            u_pad[:-2, 1:-1]  +  # North
            u_pad[2:, 1:-1]      # South
        )

        u[mask]  = u_new[mask]
        u[~mask] = image[~mask]

        max_change = float(np.max(np.abs(u[mask] - u_old[mask]))) if np.any(mask) else 0.0
        residuals.append(max_change)

        if verbose and (iteration + 1) % 1000 == 0:
            print(f"  Harmonic Iter {iteration + 1:5d} | max Δu = {max_change:.6f}")

        if max_change < tol:
            if verbose:
                print(f"  Harmonic converged at iteration {iteration + 1}")
            break
    else:
        if verbose:
            print(f"  Harmonic reached max_iter={max_iter}")

    return u, residuals

print("Harmonic defined")

### 1.2 Total Variation (TV) Inpainting

Uses conductivity **G = 1/|∇u|** (with a small ε for stability), which suppresses diffusion across edges while still filling smooth regions.

In [ ]:
def tv_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    max_iter: int = 3000,
    eps: float = 0.01,
    tol: float = 1e-5,
    verbose: bool = True,
) -> tuple:
    """TV inpainting via weighted Gauss-Seidel on the steady-state equation."""
    u = image.copy().astype(np.float64)

    dilated = binary_dilation(mask)
    boundary = dilated & ~mask
    if np.any(boundary):
        u[mask] = image[boundary].mean()
    elif np.any(~mask):
        u[mask] = image[~mask].mean()
    else:
        u[mask] = 0.5

    residuals = []
    eps_sq = eps ** 2

    for iteration in range(max_iter):
        u_old = u.copy()

        u_pad = np.pad(u, 1, mode='edge')

        # Conductivity weights at the four half-points
        diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
        diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
        diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
        diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]

        G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
        G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
        G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
        G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)

        u_E = u_pad[1:-1, 2:]
        u_W = u_pad[1:-1, :-2]
        u_N = u_pad[:-2, 1:-1]
        u_S = u_pad[2:, 1:-1]

        u_new = (G_E * u_E + G_W * u_W + G_N * u_N + G_S * u_S) / (G_E + G_W + G_N + G_S)

        u[mask]  = u_new[mask]
        u[~mask] = image[~mask]
        np.clip(u, 0.0, 1.0, out=u)

        max_change = float(np.max(np.abs(u[mask] - u_old[mask]))) if np.any(mask) else 0.0
        residuals.append(max_change)

        if verbose and (iteration + 1) % 500 == 0:
            print(f"  TV Iter {iteration + 1:5d} | max Δu = {max_change:.6f}")

        if max_change < tol:
            if verbose:
                print(f"  TV converged at iteration {iteration + 1}")
            break
    else:
        if verbose:
            print(f"  TV reached max_iter={max_iter}")

    return u, residuals

print("TV defined")

### 1.3 Curvature-Driven Diffusion (CDD) Inpainting

Based on Chan & Shen (2001). Uses **G = g(|κ|)/|∇u|** where:
- κ is the isophote curvature
- g(s) = s^p with p ≥ 1

The key property is that g(0) = 0, so diffusion halts along straight edges. A two-phase scheme is used: a warm-up pass (Harmonic or sharp TV) initialises the solution, then the CDD iteration refines it. Parameters are chosen automatically based on whether the input looks like a binary/synthetic image or a continuous photo.

In [ ]:
def _detect_image_type(image, mask):
    """Classify image as 'structure' (binary/synthetic) or 'photo' (continuous)."""
    outside_vals = image[~mask].flatten() if len(image.shape) == 3 else image[~mask]
    unique_vals = len(np.unique(np.round(outside_vals, 2)))
    return 'structure' if unique_vals <= 10 else 'photo'


def _universal_cdd_gray(image, mask, mode='auto', verbose=True):
    """Universal CDD for grayscale images."""
    if mode == 'auto':
        mode = _detect_image_type(image, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")

    dist_map = distance_transform_edt(mask)
    max_gap_radius = np.max(dist_map) if np.any(mask) else 0

    if verbose:
        print(f"Gap radius: {max_gap_radius:.1f}px")

    u = image.copy().astype(np.float64)
    dilated  = binary_dilation(mask)
    boundary = dilated & ~mask

    if mode == 'structure':
        # Mean init gives a smooth gradient for CDD to follow
        u[mask] = image[boundary].mean() if np.any(boundary) else 0.5
        if verbose:
            print("  Init: boundary mean")
    else:
        # Nearest-boundary init preserves edge structure
        if np.any(boundary):
            _, indices = distance_transform_edt(~boundary, return_indices=True)
            my, mx = np.where(mask)
            u[my, mx] = image[indices[0][my, mx], indices[1][my, mx]]
        if verbose:
            print("  Init: nearest-boundary")

    if mode == 'structure':
        if max_gap_radius <= 2.5:
            phase1_iters, sigma, p, cdd_iters = 50, 0.5, 1.0, 3000
        elif max_gap_radius <= 8.0:
            phase1_iters = int(100 * max_gap_radius)
            sigma, p, cdd_iters = 2.0, 2.0, 7000
        else:
            phase1_iters = 1500
            sigma = min(max_gap_radius / 3.0, 2.0)
            p, cdd_iters = 2.5, 10000
        eps_phase1 = None  # use Harmonic warm-up
    else:
        if max_gap_radius <= 2.0:
            phase1_iters, sigma, p, cdd_iters = 100, 0.0, 1.0, 500
        elif max_gap_radius <= 5.0:
            phase1_iters, sigma, p, cdd_iters = 300, 0.2, 1.0, 1000
        else:
            phase1_iters, sigma, p, cdd_iters = 500, 0.5, 1.0, 2000
        eps_phase1 = 0.001  # use sharp TV warm-up

    # Phase 1: warm-up (Harmonic for structure mode, sharp TV for photo mode)
    if phase1_iters > 0:
        if eps_phase1 is None:
            if verbose:
                print(f"  Phase 1: Harmonic ({phase1_iters} iters)")
            for _ in range(phase1_iters):
                u_pad = np.pad(u, 1, mode='edge')
                u_new = 0.25 * (u_pad[:-2,1:-1] + u_pad[2:,1:-1] +
                                u_pad[1:-1,:-2] + u_pad[1:-1,2:])
                u[mask] = u_new[mask]
        else:
            if verbose:
                print(f"  Phase 1: Sharp TV ({phase1_iters} iters, eps={eps_phase1})")
            eps_sq = eps_phase1 ** 2
            for _ in range(phase1_iters):
                u_pad  = np.pad(u, 1, mode='edge')
                diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
                diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
                diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
                diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]
                G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
                G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
                G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
                G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)
                u_new = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                         G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1]) / (G_E+G_W+G_N+G_S)
                u[mask] = u_new[mask]
                np.clip(u, 0, 1, out=u)

    # Phase 2: CDD iteration — G = g(|κ|) / |∇u|
    if cdd_iters > 0:
        if verbose:
            print(f"  Phase 2: CDD ({cdd_iters} iters, p={p}, σ={sigma:.1f})")
        eps_cdd = 1e-4
        dt      = 0.02
        for iteration in range(cdd_iters):
            u_old = u.copy()
            us    = gaussian_filter(u, sigma=sigma) if sigma > 0 else u

            ux  = (np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)) / 2.0
            uy  = (np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)) / 2.0
            uxx = np.roll(us, -1, axis=1) - 2.0*us + np.roll(us, 1, axis=1)
            uyy = np.roll(us, -1, axis=0) - 2.0*us + np.roll(us, 1, axis=0)
            uxy = (np.roll(ux, -1, axis=0) - np.roll(ux, 1, axis=0)) / 2.0

            grad_mag = np.sqrt(ux**2 + uy**2 + eps_cdd**2)
            kappa    = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)

            G = np.clip((np.abs(kappa) ** p) / grad_mag, 0, 10.0)

            G_pad = np.pad(G, 1, mode='edge')
            G_E = 0.5 * (G_pad[1:-1, 2:]  + G_pad[1:-1, 1:-1])
            G_W = 0.5 * (G_pad[1:-1, :-2] + G_pad[1:-1, 1:-1])
            G_N = 0.5 * (G_pad[:-2, 1:-1] + G_pad[1:-1, 1:-1])
            G_S = 0.5 * (G_pad[2:, 1:-1]  + G_pad[1:-1, 1:-1])

            u_pad = np.pad(u, 1, mode='edge')
            divergence = (G_E * (u_pad[1:-1,2:]  - u_pad[1:-1,1:-1]) +
                          G_W * (u_pad[1:-1,:-2] - u_pad[1:-1,1:-1]) +
                          G_N * (u_pad[:-2,1:-1] - u_pad[1:-1,1:-1]) +
                          G_S * (u_pad[2:,1:-1]  - u_pad[1:-1,1:-1]))

            u[mask] = u[mask] + dt * divergence[mask]
            np.clip(u, 0, 1, out=u)

            if np.max(np.abs(u[mask] - u_old[mask])) < 1e-5:
                if verbose:
                    print(f"    Converged at iter {iteration + 1}")
                break

    if mode == 'structure' and len(np.unique(image[~mask])) <= 3:
        u = np.where(u > 0.5, 1.0, 0.0)

    return u, []


def _universal_cdd_rgb(image_rgb, mask, mode='auto', verbose=True):
    """Universal CDD for RGB images — processes each channel independently."""
    if mode == 'auto':
        mode = _detect_image_type(image_rgb, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")
    result = np.zeros_like(image_rgb, dtype=np.float64)
    for c in range(3):
        result[:,:,c], _ = _universal_cdd_gray(
            image_rgb[:,:,c], mask, mode, verbose=(verbose and c == 0)
        )
    return result, []


def universal_cdd_inpaint(image, mask, mode='auto', verbose=True):
    """CDD inpainting with automatic mode detection (grayscale or RGB)."""
    if len(image.shape) == 3:
        return _universal_cdd_rgb(image, mask, mode, verbose)
    return _universal_cdd_gray(image, mask, mode, verbose)

print("CDD defined")

### 1.4 Quick CDD (QCDD) Inpainting

Removes the |∇u| denominator from CDD: **G = g(|κ|) = |κ|^p**.

This avoids the instability that arises when |∇u| → 0 near corners, giving faster and more stable convergence. The connectivity principle still holds — diffusion is steered by curvature — but the sharpening effect at edges is weaker than full CDD.

In [ ]:
def _universal_qcdd_gray(image, mask, mode='auto', verbose=True):
    """Universal QCDD for grayscale images."""
    if mode == 'auto':
        mode = _detect_image_type(image, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")

    dist_map = distance_transform_edt(mask)
    max_gap_radius = np.max(dist_map) if np.any(mask) else 0

    if verbose:
        print(f"Gap radius: {max_gap_radius:.1f}px")

    u = image.copy().astype(np.float64)
    dilated  = binary_dilation(mask)
    boundary = dilated & ~mask

    if mode == 'structure':
        u[mask] = image[boundary].mean() if np.any(boundary) else 0.5
        if verbose:
            print("  Init: boundary mean")
    else:
        if np.any(boundary):
            _, indices = distance_transform_edt(~boundary, return_indices=True)
            my, mx = np.where(mask)
            u[my, mx] = image[indices[0][my, mx], indices[1][my, mx]]
        if verbose:
            print("  Init: nearest-boundary")

    if mode == 'structure':
        if max_gap_radius <= 2.5:
            warmup, sigma, p, iters = 50, 0.5, 1.0, 3000
        elif max_gap_radius <= 8.0:
            warmup  = int(100 * max_gap_radius)
            sigma, p, iters = 2.0, 2.5, 7000
        else:
            warmup  = 1500
            sigma   = min(max_gap_radius / 3.0, 2.0)
            p, iters = 2.5, 10000
        use_tv_warmup = False
    else:
        if max_gap_radius <= 2.0:
            warmup, sigma, p, iters = 100, 0.0, 1.0, 500
        elif max_gap_radius <= 5.0:
            warmup, sigma, p, iters = 300, 0.2, 1.0, 1000
        else:
            warmup, sigma, p, iters = 500, 0.5, 1.0, 2000
        use_tv_warmup = True
        eps_tv = 0.001

    # Phase 1: warm-up
    if warmup > 0:
        if use_tv_warmup:
            if verbose:
                print(f"  Phase 1: Sharp TV ({warmup} iters)")
            eps_sq = eps_tv ** 2
            for _ in range(warmup):
                u_pad  = np.pad(u, 1, mode='edge')
                diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
                diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
                diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
                diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]
                G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
                G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
                G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
                G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)
                u_new = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                         G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1]) / (G_E+G_W+G_N+G_S)
                u[mask] = u_new[mask]
                np.clip(u, 0, 1, out=u)
        else:
            if verbose:
                print(f"  Phase 1: Harmonic ({warmup} iters)")
            for _ in range(warmup):
                u_pad = np.pad(u, 1, mode='edge')
                u_new = 0.25 * (u_pad[:-2,1:-1] + u_pad[2:,1:-1] +
                                u_pad[1:-1,:-2] + u_pad[1:-1,2:])
                u[mask] = u_new[mask]

    # Phase 2: QCDD — G = g(|κ|) = |κ|^p  (no gradient denominator)
    if verbose:
        print(f"  Phase 2: QCDD ({iters} iters, p={p}, σ={sigma})")

    eps_qcdd = 1e-4
    dt       = 0.25

    for iteration in range(iters):
        u_old = u.copy()
        us    = gaussian_filter(u, sigma=sigma) if sigma > 0 else u

        ux  = (np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)) / 2.0
        uy  = (np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)) / 2.0
        uxx = np.roll(us, -1, axis=1) - 2.0*us + np.roll(us, 1, axis=1)
        uyy = np.roll(us, -1, axis=0) - 2.0*us + np.roll(us, 1, axis=0)
        uxy = (np.roll(ux, -1, axis=0) - np.roll(ux, 1, axis=0)) / 2.0

        grad_mag = np.sqrt(ux**2 + uy**2 + eps_qcdd**2)
        kappa    = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)

        G     = np.clip(np.abs(kappa) ** p, 0, 10.0)
        G_pad = np.pad(G, 1, mode='edge')
        G_E   = 0.5 * (G_pad[1:-1, 2:]  + G_pad[1:-1, 1:-1])
        G_W   = 0.5 * (G_pad[1:-1, :-2] + G_pad[1:-1, 1:-1])
        G_N   = 0.5 * (G_pad[:-2, 1:-1] + G_pad[1:-1, 1:-1])
        G_S   = 0.5 * (G_pad[2:, 1:-1]  + G_pad[1:-1, 1:-1])

        u_pad    = np.pad(u, 1, mode='edge')
        sum_G_u  = (G_E * u_pad[1:-1, 2:]  + G_W * u_pad[1:-1, :-2] +
                    G_N * u_pad[:-2, 1:-1]  + G_S * u_pad[2:, 1:-1])
        sum_G    = G_E + G_W + G_N + G_S + 1e-10
        u_new    = (u + dt * sum_G_u) / (1.0 + dt * sum_G)

        u[mask]  = u_new[mask]
        u[~mask] = image[~mask]
        np.clip(u, 0, 1, out=u)

        if np.max(np.abs(u[mask] - u_old[mask])) < 1e-5:
            if verbose:
                print(f"    Converged at iter {iteration + 1}")
            break

    if mode == 'structure' and len(np.unique(image[~mask])) <= 3:
        u = np.where(u > 0.5, 1.0, 0.0)

    return u, []


def _universal_qcdd_rgb(image_rgb, mask, mode='auto', verbose=True):
    """Universal QCDD for RGB images."""
    if mode == 'auto':
        mode = _detect_image_type(image_rgb, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")
    result = np.zeros_like(image_rgb, dtype=np.float64)
    for c in range(3):
        result[:,:,c], _ = _universal_qcdd_gray(
            image_rgb[:,:,c], mask, mode, verbose=(verbose and c == 0)
        )
    return result, []


def universal_qcdd_inpaint(image, mask, mode='auto', verbose=True):
    """QCDD inpainting with automatic mode detection (grayscale or RGB)."""
    if len(image.shape) == 3:
        return _universal_qcdd_rgb(image, mask, mode, verbose)
    return _universal_qcdd_gray(image, mask, mode, verbose)

print("QCDD defined")

## 2. Utilities

In [ ]:
def load_image(path, grayscale=False):
    """Load an image from disk as float32 in [0, 1]."""
    mode = 'L' if grayscale else 'RGB'
    img  = Image.open(path).convert(mode)
    return np.array(img, dtype=np.float32) / 255.0


def show_images(images, titles, cmap=None, figsize=None):
    """Quick multi-panel display helper."""
    n = len(images)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap or 'gray', vmin=0, vmax=1)
        else:
            ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Helpers defined.")

## 3. Synthetic Shape Tests

These tests isolate specific behaviours: connectivity across a gap (broken bar), curve following (broken ring), and complex topology (ring with cross/X). Binary images make model behaviour easy to read at a glance.

### 3.1 Broken Bar — Connectivity Test

The simplest case: a horizontal bar with a rectangular gap cut through it. Harmonic blurs across, TV pinches off (the "chord effect"), CDD and QCDD should reconnect it cleanly.

In [ ]:
def create_broken_bar(size=64, bar_width=8, gap_width=20):
    img = np.zeros((size, size))
    img[size//2 - bar_width//2 : size//2 + bar_width//2, :] = 1.0

    mask = np.zeros((size, size), dtype=bool)
    mask[:, size//2 - gap_width//2 : size//2 + gap_width//2] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_broken_bar(size=64, bar_width=10, gap_width=12)

print("Running Harmonic...")
h_res,    _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running TV...")
tv_res,   _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running CDD...")
cdd_res,  _ = universal_cdd_inpaint(damaged, mask, verbose=True)
print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Pinches off)", "CDD\n(Connects)", "QCDD"]
for ax, img, title in zip(axes, [damaged, h_res, tv_res, cdd_res, qcdd_res], titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Broken Bar — Connectivity", fontsize=16)
plt.tight_layout()
plt.show()

### 3.2 Broken Ring — Curve Following Test

A ring with a diagonal gap in the NW quadrant. The gap width matches the ring thickness, so TV sees a locally straight boundary and draws a chord. CDD should arc through correctly.

In [ ]:
def create_broken_ring_nw_diagonal(size=100, radius=30, thickness=16, gap_length=24, gap_width=16):
    """Ring with diagonal gap in the NW quadrant."""
    Y, X      = np.ogrid[:size, :size]
    center    = size // 2
    dist      = np.sqrt((X - center)**2 + (Y - center)**2)

    img       = np.ones((size, size))
    img[(dist >= radius - thickness/2) & (dist <= radius + thickness/2)] = 0.0

    mask         = np.zeros((size, size), dtype=bool)
    angle        = np.radians(135)
    gap_center_x = center + int(radius * np.cos(angle))
    gap_center_y = center - int(radius * np.sin(angle))

    for y in range(size):
        for x in range(size):
            dx = x - gap_center_x
            dy = y - gap_center_y
            diag_parallel = (dx - dy) / np.sqrt(2)
            diag_perp     = (dx + dy) / np.sqrt(2)
            if abs(diag_parallel) < gap_length / 2 and abs(diag_perp) < gap_width / 2:
                mask[y, x] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_broken_ring_nw_diagonal(thickness=16, gap_length=16, gap_width=16)

print("Running Harmonic...")
h_res,    _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running TV...")
tv_res,   _ = tv_inpaint(damaged, mask, max_iter=5000, verbose=False)
print("Running CDD...")
cdd_res,  _ = universal_cdd_inpaint(damaged, mask, verbose=True)
print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Chord effect)", "CDD\n(Rebuilds curve)", "QCDD"]
for ax, img, title in zip(axes, [damaged, h_res, tv_res, cdd_res, qcdd_res], titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Broken Ring — Curve Following", fontsize=16)
plt.tight_layout()
plt.show()

### 3.3 Ring with Cross — Complex Topology Test

A full ring crossed by a wide cross mask. The mask cuts through the ring at four points simultaneously, requiring the model to infer both curvature and connectivity across the intersection.

In [ ]:
def create_ring_with_cross(size=100, radius=30, thickness=8, cross_thickness=14):
    img = np.ones((size, size))
    Y, X = np.ogrid[:size, :size]
    dist = np.sqrt((X - size//2)**2 + (Y - size//2)**2)
    img[(dist >= radius - thickness/2) & (dist <= radius + thickness/2)] = 0.0

    mask = np.zeros((size, size), dtype=bool)
    mask[:, size//2 - cross_thickness//2 : size//2 + cross_thickness//2] = True
    mask[size//2 - cross_thickness//2 : size//2 + cross_thickness//2, :] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_ring_with_cross()

print("Running Harmonic...")
h_res,    _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running TV...")
tv_res,   _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running CDD...")
cdd_res,  _ = universal_cdd_inpaint(damaged, mask, verbose=True)
print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Straightens/Pinches)", "CDD\n(Rebuilds ring)", "QCDD"]
for ax, img, title in zip(axes, [damaged, h_res, tv_res, cdd_res, qcdd_res], titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Ring with Cross — Complex Topology", fontsize=16)
plt.tight_layout()
plt.savefig('ring_cross_horizontal.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: ring_cross_horizontal.pdf")

### 3.4 Ring with X — Diagonal Topology Test

Same ring, but masked with a diagonal X instead of an axis-aligned cross. Diagonals are harder because the discrete grid introduces staircase artifacts that can mislead curvature estimation.

In [ ]:
def create_ring_with_x(size=100, radius=30, thickness=16, x_thickness=12):
    img = np.ones((size, size))
    Y, X = np.ogrid[:size, :size]
    center = size // 2
    dist   = np.sqrt((X - center)**2 + (Y - center)**2)
    img[(dist >= radius - thickness/2) & (dist <= radius + thickness/2)] = 0.0

    mask      = np.zeros((size, size), dtype=bool)
    half_thick = x_thickness / 2
    for y in range(size):
        for x in range(size):
            dist_nw_se = abs((x - center) - (y - center)) / np.sqrt(2)
            dist_ne_sw = abs((x - center) + (y - center)) / np.sqrt(2)
            if dist_nw_se < half_thick or dist_ne_sw < half_thick:
                mask[y, x] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_ring_with_x(thickness=16, x_thickness=12)

print("Running Harmonic...")
h_res,    _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)
print("Running TV...")
tv_res,   _ = tv_inpaint(damaged, mask, max_iter=10000, verbose=False)
print("Running CDD...")
cdd_res,  _ = universal_cdd_inpaint(damaged, mask, verbose=True)
print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

fig = plt.figure(figsize=(6, 9))
gs  = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 1], hspace=0.35, wspace=0.1)

ax_top = fig.add_subplot(gs[0, :])
ax_top.imshow(damaged, cmap='gray', vmin=0, vmax=1)
ax_top.set_title('Damaged', fontsize=18)
ax_top.axis('off')

grid_items = [
    (h_res,    'Harmonic',  1, 0),
    (tv_res,   'TV',        1, 1),
    (cdd_res,  'CDD',       2, 0),
    (qcdd_res, 'QCDD',      2, 1),
]
for img, title, row, col in grid_items:
    ax = fig.add_subplot(gs[row, col])
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=18)
    ax.axis('off')

plt.savefig('ring_x.pdf', bbox_inches='tight', dpi=150)
print("Saved: ring_x.pdf")
plt.show()

## 4. Photo Inpainting

Tests the models on standard scikit-image test images with a realistic damage mask (a rectangular block + random scratches). Metrics are PSNR and SSIM computed over the full image.

In [ ]:
from skimage import data
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

IMAGE_NAME = 'astronaut'  # options: 'astronaut', 'camera', 'chelsea', 'coffee', 'horse'

def load_test_image(name, size=256):
    loaders = {
        'astronaut': lambda: data.astronaut(),
        'camera':    lambda: np.stack([data.camera()]*3, axis=-1),
        'chelsea':   lambda: data.chelsea(),
        'coffee':    lambda: data.coffee(),
        'horse':     lambda: np.stack([(data.horse().astype(np.float32)*255).astype(np.uint8)]*3, axis=-1),
    }
    img = loaders[name]()
    return resize(img, (size, size, 3), anti_aliasing=True).astype(np.float64)

def create_photo_damage_mask(size=256, seed=42):
    """Rectangular block + a few random scratches."""
    np.random.seed(seed)
    mask = np.zeros((size, size), dtype=bool)

    block_size = size // 6
    mask[size//3 : size//3 + block_size, size//2 : size//2 + block_size] = True

    for _ in range(5):
        y       = np.random.randint(size//4, 3*size//4)
        x_start = np.random.randint(0, size//2)
        x_end   = x_start + np.random.randint(size//6, size//3)
        t       = np.random.randint(2, 5)
        mask[y:y+t, x_start:x_end] = True

    return mask

def inpaint_rgb(image, mask, method='harmonic', **kwargs):
    """Run one inpainting method on all three channels."""
    result = np.zeros_like(image)
    for c in range(3):
        if method == 'harmonic':
            result[:,:,c], _ = harmonic_inpaint(image[:,:,c], mask, **kwargs)
        elif method == 'tv':
            result[:,:,c], _ = tv_inpaint(image[:,:,c], mask, **kwargs)
        elif method == 'cdd':
            result, _ = universal_cdd_inpaint(image, mask, mode='photo', verbose=False)
            return np.clip(result, 0, 1)
        elif method == 'qcdd':
            result, _ = universal_qcdd_inpaint(image, mask, mode='photo', verbose=False)
    return np.clip(result, 0, 1)

def calculate_metrics(original, result, mask):
    p = psnr(original, result, data_range=1.0)
    s = ssim(original, result, data_range=1.0, channel_axis=2)
    return p, s

original = load_test_image(IMAGE_NAME, size=256)
mask     = create_photo_damage_mask(size=256)
damaged  = original.copy()
damaged[mask] = 0.5

print(f"Image: {IMAGE_NAME} (256x256)")
print(f"Mask:  {mask.sum()} pixels ({100*mask.mean():.1f}%)")

print("\nRunning Harmonic...")
harmonic_res = inpaint_rgb(damaged, mask, method='harmonic', max_iter=3000, verbose=False)
print("Running TV...")
tv_res = inpaint_rgb(damaged, mask, method='tv', max_iter=3000, verbose=False)
print("Running CDD...")
cdd_res = inpaint_rgb(damaged, mask, method='cdd')
print("Running QCDD...")
qcdd_res = inpaint_rgb(damaged, mask, method='qcdd')

results = {'Harmonic': harmonic_res, 'TV': tv_res, 'CDD': cdd_res, 'QCDD': qcdd_res}
metrics = {}
print(f"\n{'Model':<12} {'PSNR (dB)':<12} {'SSIM'}")
print("-" * 36)
for name, result in results.items():
    p, s = calculate_metrics(original, result, mask)
    metrics[name] = {'psnr': p, 'ssim': s}
    print(f"{name:<12} {p:<12.2f} {s:.4f}")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0,0].imshow(original);      axes[0,0].set_title("Original", fontsize=12)
axes[0,1].imshow(damaged);       axes[0,1].set_title("Damaged",  fontsize=12)
mask_overlay = original.copy();  mask_overlay[mask] = [1, 0, 0]
axes[0,2].imshow(mask_overlay);  axes[0,2].set_title("Mask Region", fontsize=12)
axes[1,0].imshow(harmonic_res);  axes[1,0].set_title(f"Harmonic\nPSNR: {metrics['Harmonic']['psnr']:.2f} | SSIM: {metrics['Harmonic']['ssim']:.4f}", fontsize=11)
axes[1,1].imshow(tv_res);        axes[1,1].set_title(f"TV\nPSNR: {metrics['TV']['psnr']:.2f} | SSIM: {metrics['TV']['ssim']:.4f}", fontsize=11)
axes[1,2].imshow(cdd_res);       axes[1,2].set_title(f"CDD\nPSNR: {metrics['CDD']['psnr']:.2f} | SSIM: {metrics['CDD']['ssim']:.4f}", fontsize=11)

for ax in axes.flat:
    ax.axis('off')
plt.suptitle(f"Photo Inpainting: {IMAGE_NAME}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Grid Search — CDD and QCDD

Searches over warm-up iterations, CDD/QCDD iterations, p, σ, and (for QCDD) the time step dt. Evaluates each configuration by PSNR and SSIM on a single grayscale image.

In [ ]:
import itertools
from skimage import data as skdata
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

IMAGE_NAME  = 'camera'  # options: 'camera', 'coins', 'moon', 'brick'
IMAGE_SIZE  = 256
SEED        = 42

def load_grayscale_image(name, size=256):
    loaders = {
        'camera': lambda: skdata.camera(),
        'coins':  lambda: skdata.coins(),
        'moon':   lambda: skdata.moon(),
        'brick':  lambda: skdata.brick(),
    }
    img = resize(loaders[name](), (size, size), anti_aliasing=True)
    return img.astype(np.float64)

def create_damage_mask(size=256, seed=42):
    np.random.seed(seed)
    mask = np.zeros((size, size), dtype=bool)
    start = size // 3
    mask[start : start + size//5, start : start + size//5] = True
    for _ in range(4):
        y       = np.random.randint(size//4, 3*size//4)
        x_start = np.random.randint(0, size//2)
        x_end   = x_start + np.random.randint(size//8, size//4)
        t       = np.random.randint(2, 4)
        mask[y:y+t, x_start:x_end] = True
    return mask

def cdd_gray_configurable(image, mask, phase1_iters=300, cdd_iters=1000,
                           p=1.0, sigma=0.5, eps_tv=0.001, dt=0.02):
    """CDD for grayscale with all hyperparameters exposed."""
    u = image.copy().astype(np.float64)
    dilated  = binary_dilation(mask)
    boundary = dilated & ~mask
    if np.any(boundary):
        _, indices = distance_transform_edt(~boundary, return_indices=True)
        my, mx = np.where(mask)
        u[my, mx] = image[indices[0][my, mx], indices[1][my, mx]]

    eps_sq = eps_tv ** 2
    for _ in range(phase1_iters):
        u_pad  = np.pad(u, 1, mode='edge')
        diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
        diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
        diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
        diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]
        G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
        G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
        G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
        G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)
        u_new = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                 G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1]) / (G_E+G_W+G_N+G_S)
        u[mask] = u_new[mask]
        np.clip(u, 0, 1, out=u)

    eps_cdd = 1e-4
    for _ in range(cdd_iters):
        u_old = u.copy()
        us    = gaussian_filter(u, sigma=sigma) if sigma > 0 else u
        ux  = (np.roll(us,-1,axis=1) - np.roll(us,1,axis=1)) / 2.0
        uy  = (np.roll(us,-1,axis=0) - np.roll(us,1,axis=0)) / 2.0
        uxx = np.roll(us,-1,axis=1) - 2.0*us + np.roll(us,1,axis=1)
        uyy = np.roll(us,-1,axis=0) - 2.0*us + np.roll(us,1,axis=0)
        uxy = (np.roll(ux,-1,axis=0) - np.roll(ux,1,axis=0)) / 2.0
        grad_mag = np.sqrt(ux**2 + uy**2 + eps_cdd**2)
        kappa    = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)
        G     = np.clip((np.abs(kappa)**p) / grad_mag, 0, 10.0)
        G_pad = np.pad(G, 1, mode='edge')
        G_E   = 0.5*(G_pad[1:-1,2:]  + G_pad[1:-1,1:-1])
        G_W   = 0.5*(G_pad[1:-1,:-2] + G_pad[1:-1,1:-1])
        G_N   = 0.5*(G_pad[:-2,1:-1] + G_pad[1:-1,1:-1])
        G_S   = 0.5*(G_pad[2:,1:-1]  + G_pad[1:-1,1:-1])
        u_pad = np.pad(u, 1, mode='edge')
        div   = (G_E*(u_pad[1:-1,2:] -u_pad[1:-1,1:-1]) + G_W*(u_pad[1:-1,:-2]-u_pad[1:-1,1:-1]) +
                 G_N*(u_pad[:-2,1:-1]-u_pad[1:-1,1:-1]) + G_S*(u_pad[2:,1:-1] -u_pad[1:-1,1:-1]))
        u[mask] = u[mask] + dt * div[mask]
        np.clip(u, 0, 1, out=u)
        if np.max(np.abs(u[mask] - u_old[mask])) < 1e-6:
            break
    return u


def qcdd_gray_configurable(image, mask, phase1_iters=300, qcdd_iters=1000,
                            p=1.0, sigma=0.5, eps_tv=0.001, dt=0.25):
    """QCDD for grayscale with all hyperparameters exposed."""
    u = image.copy().astype(np.float64)
    dilated  = binary_dilation(mask)
    boundary = dilated & ~mask
    if np.any(boundary):
        _, indices = distance_transform_edt(~boundary, return_indices=True)
        my, mx = np.where(mask)
        u[my, mx] = image[indices[0][my, mx], indices[1][my, mx]]

    eps_sq = eps_tv ** 2
    for _ in range(phase1_iters):
        u_pad  = np.pad(u, 1, mode='edge')
        diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
        diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
        diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
        diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]
        G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
        G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
        G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
        G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)
        u_new = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                 G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1]) / (G_E+G_W+G_N+G_S)
        u[mask] = u_new[mask]
        np.clip(u, 0, 1, out=u)

    eps_qcdd = 1e-4
    for _ in range(qcdd_iters):
        u_old = u.copy()
        us    = gaussian_filter(u, sigma=sigma) if sigma > 0 else u
        ux  = (np.roll(us,-1,axis=1) - np.roll(us,1,axis=1)) / 2.0
        uy  = (np.roll(us,-1,axis=0) - np.roll(us,1,axis=0)) / 2.0
        uxx = np.roll(us,-1,axis=1) - 2.0*us + np.roll(us,1,axis=1)
        uyy = np.roll(us,-1,axis=0) - 2.0*us + np.roll(us,1,axis=0)
        uxy = (np.roll(ux,-1,axis=0) - np.roll(ux,1,axis=0)) / 2.0
        grad_mag = np.sqrt(ux**2 + uy**2 + eps_qcdd**2)
        kappa    = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)
        G     = np.clip(np.abs(kappa)**p, 0, 10.0)
        G_pad = np.pad(G, 1, mode='edge')
        G_E   = 0.5*(G_pad[1:-1,2:]  + G_pad[1:-1,1:-1])
        G_W   = 0.5*(G_pad[1:-1,:-2] + G_pad[1:-1,1:-1])
        G_N   = 0.5*(G_pad[:-2,1:-1] + G_pad[1:-1,1:-1])
        G_S   = 0.5*(G_pad[2:,1:-1]  + G_pad[1:-1,1:-1])
        u_pad   = np.pad(u, 1, mode='edge')
        sum_G_u = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                   G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1])
        sum_G   = G_E + G_W + G_N + G_S + 1e-10
        u_new   = (u + dt * sum_G_u) / (1.0 + dt * sum_G)
        u[mask] = u_new[mask]
        np.clip(u, 0, 1, out=u)
        if np.max(np.abs(u[mask] - u_old[mask])) < 1e-6:
            break
    return u


param_grid = {
    'phase1_iters': [100, 300, 500],
    'main_iters':   [1500, 2000, 3000],
    'p':            [0.8, 1.0, 1.2],
    'sigma':        [0.0, 0.3, 0.5],
    'eps_tv':       [0.001],
}
dt_values = [0.4, 0.5, 0.6]

print(f"Loading {IMAGE_NAME} ({IMAGE_SIZE}x{IMAGE_SIZE})")
original = load_grayscale_image(IMAGE_NAME, IMAGE_SIZE)
mask     = create_damage_mask(IMAGE_SIZE, SEED)
damaged  = original.copy()
damaged[mask] = 0.5
print(f"Mask: {mask.sum()} pixels ({100*mask.mean():.1f}%)")

keys         = list(param_grid.keys())
combinations = list(itertools.product(*[param_grid[k] for k in keys]))
print(f"\nCDD configurations:  {len(combinations)}")
print(f"QCDD configurations: {len(combinations) * len(dt_values)}")

cdd_results  = []
qcdd_results = []

print("\nRunning CDD grid search...")
for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(combinations)}")
    try:
        result  = cdd_gray_configurable(damaged, mask,
                      phase1_iters=params['phase1_iters'], cdd_iters=params['main_iters'],
                      p=params['p'], sigma=params['sigma'], eps_tv=params['eps_tv'], dt=0.02)
        cdd_results.append({'params': params,
                             'psnr': psnr(original, result, data_range=1.0),
                             'ssim': ssim(original, result, data_range=1.0)})
    except Exception as e:
        print(f"  CDD failed {params}: {e}")

print("\nRunning QCDD grid search...")
counter = 0
for combo in combinations:
    params = dict(zip(keys, combo))
    for dt in dt_values:
        counter += 1
        if counter % 30 == 0:
            print(f"  {counter}/{len(combinations)*len(dt_values)}")
        try:
            result = qcdd_gray_configurable(damaged, mask,
                         phase1_iters=params['phase1_iters'], qcdd_iters=params['main_iters'],
                         p=params['p'], sigma=params['sigma'], eps_tv=params['eps_tv'], dt=dt)
            qcdd_results.append({'params': {**params, 'dt': dt},
                                  'psnr': psnr(original, result, data_range=1.0),
                                  'ssim': ssim(original, result, data_range=1.0)})
        except Exception as e:
            print(f"  QCDD failed {params}, dt={dt}: {e}")

best_cdd_psnr  = max(cdd_results,  key=lambda x: x['psnr'])
best_cdd_ssim  = max(cdd_results,  key=lambda x: x['ssim'])
best_qcdd_psnr = max(qcdd_results, key=lambda x: x['psnr'])
best_qcdd_ssim = max(qcdd_results, key=lambda x: x['ssim'])

print("\nBest CDD (PSNR):  ", best_cdd_psnr['psnr'],  best_cdd_psnr['params'])
print("Best CDD (SSIM):  ", best_cdd_ssim['ssim'],  best_cdd_ssim['params'])
print("Best QCDD (PSNR): ", best_qcdd_psnr['psnr'], best_qcdd_psnr['params'])
print("Best QCDD (SSIM): ", best_qcdd_ssim['ssim'], best_qcdd_ssim['params'])

print("\nTop 5 CDD (by PSNR):")
for i, r in enumerate(sorted(cdd_results, key=lambda x: x['psnr'], reverse=True)[:5]):
    print(f"  {i+1}. PSNR={r['psnr']:.4f} SSIM={r['ssim']:.4f}  {r['params']}")

print("\nTop 5 QCDD (by PSNR):")
for i, r in enumerate(sorted(qcdd_results, key=lambda x: x['psnr'], reverse=True)[:5]):
    print(f"  {i+1}. PSNR={r['psnr']:.4f} SSIM={r['ssim']:.4f}  {r['params']}")

## 6. Quantitative Comparison — skimage Test Images

Runs all four models on five standard test images (astronaut, camera, chelsea, coffee, horse) with the realistic damage mask, then averages PSNR and SSIM across them.

In [ ]:
from skimage import data as skdata
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

IMAGE_SIZE = 256
SEED       = 42

def load_rgb_image(name, size=256):
    loaders = {
        'astronaut': lambda: skdata.astronaut(),
        'camera':    lambda: np.stack([skdata.camera()]*3, axis=-1),
        'chelsea':   lambda: skdata.chelsea(),
        'coffee':    lambda: skdata.coffee(),
        'horse':     lambda: np.stack([(skdata.horse().astype(np.float32)*255).astype(np.uint8)]*3, axis=-1),
    }
    img = resize(loaders[name](), (size, size, 3), anti_aliasing=True)
    return img.astype(np.float64)

def create_realistic_damage_mask(size=256, seed=42):
    """Rectangle (~1.2% of pixels) + directional scratches (~0.8%) for ~2% total."""
    np.random.seed(seed)
    mask = np.zeros((size, size), dtype=bool)

    rh, rw = 20, 50
    ry, rx = size//3, size//3
    mask[ry:ry+rh, rx:rx+rw] = True

    for (y, x0, length, t) in [
        (size//5,          size//8, size//2, 2),
        (size//2,          size//6, size//3, 4),
        (2*size//3 + 10,   size//4, size//4, 2),
    ]:
        mask[y:y+t, x0:x0+length] = True

    y0, x0 = size//2 - 10, size//2 + 20
    for k in range(size//5):
        yy, xx = y0 + k//6, x0 + k
        if 0 <= yy+4 < size and 0 <= xx < size:
            mask[yy:yy+4, xx:xx+1] = True

    xc, y0c = size//4, size//5
    mask[y0c : y0c + size//7, xc:xc+3] = True
    return mask

IMAGE_NAMES = ['astronaut', 'camera', 'chelsea', 'coffee', 'horse']
all_metrics = {m: {'psnr': [], 'ssim': []} for m in ['Harmonic', 'TV', 'CDD', 'QCDD']}

for img_name in IMAGE_NAMES:
    print(f"\n--- {img_name} ---")
    original = load_rgb_image(img_name, IMAGE_SIZE)
    mask     = create_realistic_damage_mask(IMAGE_SIZE, SEED)
    damaged  = original.copy()
    damaged[mask] = 0.5
    print(f"  Mask: {mask.sum()} px ({100*mask.mean():.2f}%)")

    print("  Harmonic...")
    h = np.clip(np.stack([harmonic_inpaint(damaged[:,:,c], mask, max_iter=3000, verbose=False)[0]
                           for c in range(3)], axis=-1), 0, 1)
    print("  TV...")
    t = np.clip(np.stack([tv_inpaint(damaged[:,:,c], mask, max_iter=8000, verbose=False)[0]
                           for c in range(3)], axis=-1), 0, 1)
    print("  CDD...")
    cdd_r,  _ = universal_cdd_inpaint(damaged, mask, mode='photo', verbose=False)
    cdd_r     = np.clip(cdd_r, 0, 1)
    print("  QCDD...")
    qcdd_r, _ = universal_qcdd_inpaint(damaged, mask, mode='photo', verbose=False)
    qcdd_r    = np.clip(qcdd_r, 0, 1)

    for mname, result in [('Harmonic', h), ('TV', t), ('CDD', cdd_r), ('QCDD', qcdd_r)]:
        p = psnr(original, result, data_range=1.0)
        s = ssim(original, result, data_range=1.0, channel_axis=2)
        all_metrics[mname]['psnr'].append(p)
        all_metrics[mname]['ssim'].append(s)
        print(f"    {mname:10s}  PSNR={p:.2f}  SSIM={s:.4f}")

models   = ['Harmonic', 'TV', 'CDD', 'QCDD']
avg_psnr = [np.mean(all_metrics[m]['psnr']) for m in models]
avg_ssim = [np.mean(all_metrics[m]['ssim']) for m in models]
std_psnr = [np.std(all_metrics[m]['psnr'])  for m in models]
std_ssim = [np.std(all_metrics[m]['ssim'])  for m in models]

print(f"\n{'Model':<12} {'Avg PSNR':>10} {'±':>4} {'Avg SSIM':>10} {'±':>4}")
print("-" * 48)
for m, p, sp, s, ss in zip(models, avg_psnr, std_psnr, avg_ssim, std_ssim):
    print(f"{m:<12} {p:>10.2f} {sp:>4.2f} {s:>10.4f} {ss:>4.4f}")

colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
x      = np.arange(len(models))
width  = 0.55

fig1, ax1 = plt.subplots(figsize=(6, 5))
bars1 = ax1.bar(x, avg_psnr, width, color=colors, edgecolor='black', linewidth=0.6,
                yerr=std_psnr, capsize=4, error_kw=dict(elinewidth=1))
ax1.set_title('Average PSNR (dB)', fontsize=13, fontweight='bold', pad=10)
ax1.set_xticks(x); ax1.set_xticklabels(models, fontsize=11)
ax1.set_ylabel('PSNR (dB)', fontsize=11)
ax1.set_ylim(min(avg_psnr) - max(std_psnr) - 0.5, max(avg_psnr) + max(std_psnr) + 0.8)
for bar, val in zip(bars1, avg_psnr):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(std_psnr) + 0.15,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.yaxis.grid(True, linestyle='--', alpha=0.5); ax1.set_axisbelow(True)
fig1.tight_layout()
fig1.savefig('psnr_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show(); print("Saved: psnr_comparison.pdf")

fig2, ax2 = plt.subplots(figsize=(6, 5))
bars2 = ax2.bar(x, avg_ssim, width, color=colors, edgecolor='black', linewidth=0.6,
                yerr=std_ssim, capsize=4, error_kw=dict(elinewidth=1))
ax2.set_title('Average SSIM', fontsize=13, fontweight='bold', pad=10)
ax2.set_xticks(x); ax2.set_xticklabels(models, fontsize=11)
ax2.set_ylabel('SSIM', fontsize=11)
ax2.set_ylim(min(avg_ssim) - max(std_ssim) - 0.005, max(avg_ssim) + max(std_ssim) + 0.008)
for bar, val in zip(bars2, avg_ssim):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(std_ssim) + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.yaxis.grid(True, linestyle='--', alpha=0.5); ax2.set_axisbelow(True)
fig2.tight_layout()
fig2.savefig('ssim_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show(); print("Saved: ssim_comparison.pdf")

# Per-model qualitative PDFs for the camera image
DEMO_IMAGE    = 'camera'
demo_original = load_rgb_image(DEMO_IMAGE, IMAGE_SIZE)
demo_mask     = create_realistic_damage_mask(IMAGE_SIZE, SEED)
demo_damaged  = demo_original.copy()
demo_damaged[demo_mask] = 0.5

print(f"\nGenerating qualitative figures for {DEMO_IMAGE}...")
demo_h = np.clip(np.stack([harmonic_inpaint(demo_damaged[:,:,c], demo_mask, max_iter=5000, verbose=False)[0]
                            for c in range(3)], axis=-1), 0, 1)
demo_t = np.clip(np.stack([tv_inpaint(demo_damaged[:,:,c], demo_mask, max_iter=5000, verbose=False)[0]
                            for c in range(3)], axis=-1), 0, 1)
demo_cdd,  _ = universal_cdd_inpaint(demo_damaged, demo_mask, mode='photo', verbose=False)
demo_qcdd, _ = universal_qcdd_inpaint(demo_damaged, demo_mask, mode='photo', verbose=False)
demo_cdd  = np.clip(demo_cdd,  0, 1)
demo_qcdd = np.clip(demo_qcdd, 0, 1)

def rgb_metrics(orig, result):
    return psnr(orig, result, data_range=1.0), ssim(orig, result, data_range=1.0, channel_axis=2)

ph,   sh   = rgb_metrics(demo_original, demo_h)
pt,   st   = rgb_metrics(demo_original, demo_t)
pcdd, scdd = rgb_metrics(demo_original, demo_cdd)
pq,   sq   = rgb_metrics(demo_original, demo_qcdd)

mask_overlay = demo_original.copy()
mask_overlay[demo_mask] = [1.0, 0.0, 0.0]

items = [
    (mask_overlay, f'Damaged ({100*demo_mask.mean():.1f}% masked)', 'damaged',  None,  None),
    (demo_h,       'Harmonic',                                       'harmonic', ph,    sh),
    (demo_t,       'TV',                                             'tv',       pt,    st),
    (demo_cdd,     'CDD',                                            'cdd',      pcdd,  scdd),
    (demo_qcdd,    'QCDD',                                           'qcdd',     pq,    sq),
]
for img, title, fname, p_val, s_val in items:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    if p_val is not None:
        ax.set_title(f'{title}\nPSNR: {p_val:.2f} dB  |  SSIM: {s_val:.4f}', fontsize=11, fontweight='bold', pad=8)
    else:
        ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{DEMO_IMAGE}_{fname}.pdf', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved: {DEMO_IMAGE}_{fname}.pdf")

## 7. Large-Scale Evaluation — Urban100 and LFW Faces

Evaluates all models on two 100-image datasets:
- **Urban100** — architectural images with strong edges and repeated structures
- **LFW Faces** — portrait photos with smooth gradients and fine texture

All images are converted to grayscale and resized to 256×256. The same damage mask is applied to every image.

> **Colab:** mount your Drive and add the model path before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/pde_v2')

In [ ]:
from skimage.color import rgb2gray
from skimage.io import imread
from skimage import data as skdata
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import os

IMAGE_SIZE = 256
SEED       = 42

mask = create_realistic_damage_mask(IMAGE_SIZE, SEED)
print(f"Mask: {mask.sum()} px ({100*mask.mean():.2f}%)")

def load_urban100(folder, size=256, n=100):
    images, names = [], []
    for i in range(1, n + 1):
        fname = f"img_{i:03d}_SRF_2_LR.png"
        fpath = os.path.join(folder, fname)
        if not os.path.exists(fpath):
            print(f"  Not found: {fname}"); continue
        try:
            img = imread(fpath)
            if img.ndim == 3:
                img = rgb2gray(img)
            img = img.astype(np.float64)
            if img.max() > 1.0:
                img /= 255.0
            img = resize(img, (size, size), anti_aliasing=True)
            images.append(img); names.append(fname)
        except Exception as e:
            print(f"  Skipping {fname}: {e}")
    print(f"Urban100 loaded: {len(images)} images")
    return images, names

def load_lfw_faces(size=256, n=100):
    images, names = [], []
    try:
        faces   = skdata.lfw_subset()
        n_faces = min(n, faces.shape[0])
        print(f"LFW subset: using {n_faces} of {faces.shape[0]} available")
        for i in range(n_faces):
            face = faces[i].astype(np.float64)
            if face.max() > 1.0:
                face /= 255.0
            face = resize(face, (size, size), anti_aliasing=True)
            images.append(face); names.append(f'lfw_{i:03d}')
    except Exception as e:
        print(f"  LFW unavailable: {e}")
    print(f"LFW loaded: {len(images)} images")
    return images, names

URBAN_FOLDER = '/content/drive/MyDrive/LOW X2 Urban'
urban_images, urban_names = load_urban100(URBAN_FOLDER, size=IMAGE_SIZE, n=100)
lfw_images,   lfw_names   = load_lfw_faces(size=IMAGE_SIZE, n=100)

def evaluate_dataset(images, names, mask, label='dataset'):
    metrics = {m: {'psnr': [], 'ssim': []} for m in ['Harmonic', 'TV', 'CDD', 'QCDD']}
    for idx, (original, img_name) in enumerate(zip(images, names)):
        if original.shape != (IMAGE_SIZE, IMAGE_SIZE):
            print(f"  Skipping {img_name}: shape {original.shape}"); continue
        damaged = original.copy(); damaged[mask] = 0.5
        if (idx + 1) % 20 == 0 or idx == 0:
            print(f"  [{label}] [{idx+1}/{len(images)}] {img_name}")

        h,    _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)
        t,    _ = tv_inpaint(damaged, mask, max_iter=3000, verbose=False)
        cdd,  _ = universal_cdd_inpaint(damaged, mask, mode='photo', verbose=False)
        qcdd, _ = universal_qcdd_inpaint(damaged, mask, mode='photo', verbose=False)

        for mname, result in [('Harmonic', np.clip(h,0,1)), ('TV', np.clip(t,0,1)),
                               ('CDD', np.clip(cdd,0,1)), ('QCDD', np.clip(qcdd,0,1))]:
            metrics[mname]['psnr'].append(psnr(original, result, data_range=1.0))
            metrics[mname]['ssim'].append(ssim(original, result, data_range=1.0))
    return metrics

print("\nRunning Urban100...")
urban_metrics = evaluate_dataset(urban_images, urban_names, mask, label='Urban100')
print("\nRunning LFW Faces...")
lfw_metrics   = evaluate_dataset(lfw_images,   lfw_names,   mask, label='LFW')

models = ['Harmonic', 'TV', 'CDD', 'QCDD']
for label, metrics in [('Urban100', urban_metrics), ('LFW Faces', lfw_metrics)]:
    avg_psnr = [np.mean(metrics[m]['psnr']) for m in models]
    avg_ssim = [np.mean(metrics[m]['ssim']) for m in models]
    std_psnr = [np.std(metrics[m]['psnr'])  for m in models]
    std_ssim = [np.std(metrics[m]['ssim'])  for m in models]
    print(f"\n{label}  (n={len(metrics['Harmonic']['psnr'])})")
    print(f"{'Model':<12} {'Avg PSNR':>10} {'±':>6} {'Avg SSIM':>10} {'±':>8}")
    print("-" * 52)
    for m, p, sp, s, ss in zip(models, avg_psnr, std_psnr, avg_ssim, std_ssim):
        print(f"{m:<12} {p:>10.2f} {sp:>6.2f} {s:>10.4f} {ss:>8.4f}")

In [ ]:
def save_bar_charts(metrics, label, suffix):
    models   = ['Harmonic', 'TV', 'CDD', 'QCDD']
    avg_psnr = [np.mean(metrics[m]['psnr']) for m in models]
    avg_ssim = [np.mean(metrics[m]['ssim']) for m in models]
    std_psnr = [np.std(metrics[m]['psnr'])  for m in models]
    std_ssim = [np.std(metrics[m]['ssim'])  for m in models]
    colors   = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
    x, width = np.arange(len(models)), 0.55

    for metric_data, std_data, ylabel, title, fname in [
        (avg_psnr, std_psnr, 'PSNR (dB)', f'Average PSNR (dB)\n{label}', f'psnr_comparison_{suffix}.pdf'),
        (avg_ssim, std_ssim, 'SSIM',      f'Average SSIM\n{label}',       f'ssim_comparison_{suffix}.pdf'),
    ]:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.bar(x, metric_data, width, color=colors, edgecolor='black', linewidth=0.6,
               yerr=std_data, capsize=4, error_kw=dict(elinewidth=1))
        ax.set_title(title, fontsize=18, fontweight='bold', pad=10)
        ax.set_xticks(x); ax.set_xticklabels(models, fontsize=18)
        ax.set_ylabel(ylabel, fontsize=18)
        ax.tick_params(axis='y', labelsize=14)
        ax.set_ylim(min(metric_data) - max(std_data) - (0.5 if 'PSNR' in ylabel else 0.005),
                    max(metric_data) + max(std_data) + (0.8 if 'PSNR' in ylabel else 0.008))
        ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight', dpi=150)
        plt.show(); print(f"Saved: {fname}")

save_bar_charts(urban_metrics, 'Urban100',  'urban')
save_bar_charts(lfw_metrics,   'LFW Faces', 'faces')

In [ ]:
def save_box_plots(metrics, label, suffix):
    models   = ['Harmonic', 'TV', 'CDD', 'QCDD']
    colors   = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
    box_kw   = dict(patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, markeredgecolor='gray', alpha=0.5))

    for data_key, ylabel, title, fname in [
        ('psnr', 'PSNR (dB)', f'PSNR Distribution (dB)\n{label}', f'psnr_boxplot_{suffix}.pdf'),
        ('ssim', 'SSIM',      f'SSIM Distribution\n{label}',       f'ssim_boxplot_{suffix}.pdf'),
    ]:
        data = [metrics[m][data_key] for m in models]
        fig, ax = plt.subplots(figsize=(6, 5))
        bp = ax.boxplot(data, **box_kw)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color); patch.set_alpha(0.8)
        ax.set_title(title, fontsize=18, fontweight='bold', pad=10)
        ax.set_xticks(range(1, len(models)+1)); ax.set_xticklabels(models, fontsize=18)
        ax.set_ylabel(ylabel, fontsize=18)
        ax.tick_params(axis='y', labelsize=14)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight', dpi=150)
        plt.show(); print(f"Saved: {fname}")

save_box_plots(urban_metrics, 'Urban100',  'urban')
save_box_plots(lfw_metrics,   'LFW Faces', 'faces')

In [ ]:
from matplotlib.patches import Patch

models      = ['Harmonic', 'TV', 'CDD', 'QCDD']
n_models    = len(models)
color_urban = '#4c72b0'
color_lfw   = '#dd8452'

def save_combined_boxplot(metric_key, ylabel, title, fname, legend_loc='upper left'):
    fig, ax = plt.subplots(figsize=(8, 5))

    positions_urban = np.arange(1, n_models * 3, 3)
    positions_lfw   = positions_urban + 1

    box_kw = dict(patch_artist=True, widths=0.8,
                  medianprops=dict(color='black', linewidth=2),
                  whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
                  flierprops=dict(marker='o', markersize=3, markeredgecolor='gray', alpha=0.5))

    bp1 = ax.boxplot([urban_metrics[m][metric_key] for m in models], positions=positions_urban, **box_kw)
    for patch in bp1['boxes']:
        patch.set_facecolor(color_urban); patch.set_alpha(0.85)

    bp2 = ax.boxplot([lfw_metrics[m][metric_key] for m in models], positions=positions_lfw, **box_kw)
    for patch in bp2['boxes']:
        patch.set_facecolor(color_lfw); patch.set_alpha(0.85)

    ax.set_xticks(positions_urban + 0.5)
    ax.set_xticklabels(models, fontsize=18)
    ax.set_xlim(0, n_models * 3)
    ax.set_title(title, fontsize=18, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.tick_params(axis='y', labelsize=14)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
    ax.legend(handles=[Patch(facecolor=color_urban, alpha=0.85, label='Urban100'),
                        Patch(facecolor=color_lfw,   alpha=0.85, label='LFW Faces')],
              fontsize=13, loc=legend_loc, prop=dict(weight='bold', size=13))
    fig.tight_layout()
    fig.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show(); print(f"Saved: {fname}")

save_combined_boxplot('psnr', 'PSNR (dB)', 'PSNR Distribution (dB)', 'psnr_boxplot_combined.pdf', legend_loc='upper left')
save_combined_boxplot('ssim', 'SSIM',      'SSIM Distribution',       'ssim_boxplot_combined.pdf', legend_loc='lower right')

## 8. Astronaut Demo

Qualitative per-model outputs for the astronaut image, saved as individual PDFs for use in the paper.

In [ ]:
IMAGE_NAME = 'astronaut'

original = load_test_image(IMAGE_NAME, size=256)
mask     = create_photo_damage_mask(size=256, seed=42)
damaged  = original.copy()
damaged[mask] = 0.5

print(f"Image: {IMAGE_NAME} (256x256)")
print(f"Mask:  {mask.sum()} pixels ({100*mask.mean():.1f}%)")

print("\nRunning Harmonic...")
harmonic_res = inpaint_rgb(damaged, mask, method='harmonic', max_iter=3000, verbose=False)
print("Running TV...")
tv_res = inpaint_rgb(damaged, mask, method='tv', max_iter=3000, verbose=False)
print("Running CDD...")
cdd_res = inpaint_rgb(damaged, mask, method='cdd')
print("Running QCDD...")
qcdd_res = inpaint_rgb(damaged, mask, method='qcdd')

metrics = {}
for name, result in [('Harmonic', harmonic_res), ('TV', tv_res), ('CDD', cdd_res), ('QCDD', qcdd_res)]:
    p, s = calculate_metrics(original, result, mask)
    metrics[name] = {'psnr': p, 'ssim': s}
    print(f"{name:<12} PSNR={p:.2f}  SSIM={s:.4f}")

mask_overlay = original.copy()
mask_overlay[mask] = [1, 0, 0]

items = [
    (mask_overlay,  f'Damaged ({100*mask.mean():.1f}% masked)', 'damaged',  None, None),
    (harmonic_res,  'Harmonic', 'harmonic', metrics['Harmonic']['psnr'], metrics['Harmonic']['ssim']),
    (tv_res,        'TV',       'tv',       metrics['TV']['psnr'],       metrics['TV']['ssim']),
    (cdd_res,       'CDD',      'cdd',      metrics['CDD']['psnr'],      metrics['CDD']['ssim']),
    (qcdd_res,      'QCDD',     'qcdd',     metrics['QCDD']['psnr'],     metrics['QCDD']['ssim']),
]
for img, title, fname, p_val, s_val in items:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    if p_val is not None:
        ax.set_title(f'{title}\nPSNR: {p_val:.2f} dB  |  SSIM: {s_val:.4f}',
                     fontsize=18, fontweight='bold', pad=8)
    else:
        ax.set_title(title, fontsize=18, fontweight='bold', pad=8)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{IMAGE_NAME}_{fname}.pdf', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved: {IMAGE_NAME}_{fname}.pdf")

## 9. Manual Inpainting on Custom Photos

Paint a damage mask directly on a photo using an HTML canvas widget, then run all four models on it.

> **Colab:** re-mount Drive and add the model path if running this section standalone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/pde_v2')

### 9.1 Dataset Sample Grids

Save representative sample images from each dataset as PDFs for inclusion in the paper.

In [ ]:
from skimage import data as skdata
from skimage.transform import resize
from skimage.io import imread
from skimage.color import rgb2gray
import os

IMAGE_SIZE   = 256
URBAN_FOLDER = '/content/drive/MyDrive/LOW X2 Urban'

# LFW Faces — 3×2 grid
faces = skdata.lfw_subset()
fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for idx, ax in enumerate(axes.flat):
    face = faces[idx].astype(np.float64)
    if face.max() > 1.0:
        face /= 255.0
    face = resize(face, (IMAGE_SIZE, IMAGE_SIZE), anti_aliasing=True)
    ax.imshow(face, cmap='gray', vmin=0, vmax=1); ax.axis('off')
plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.savefig('sample_lfw_grid.pdf', bbox_inches='tight', pad_inches=0, dpi=150)
plt.show(); print("Saved: sample_lfw_grid.pdf")

# Urban100 — 3×2 grid
urban_indices = [1, 10, 25, 40, 55, 70]
fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for idx, ax in enumerate(axes.flat):
    fpath = os.path.join(URBAN_FOLDER, f"img_{urban_indices[idx]:03d}_SRF_2_LR.png")
    img   = imread(fpath)
    if img.ndim == 3:
        img = rgb2gray(img)
    img = img.astype(np.float64)
    if img.max() > 1.0:
        img /= 255.0
    img = resize(img, (IMAGE_SIZE, IMAGE_SIZE), anti_aliasing=True)
    ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.axis('off')
plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.savefig('sample_urban_grid.pdf', bbox_inches='tight', pad_inches=0, dpi=150)
plt.show(); print("Saved: sample_urban_grid.pdf")

### 9.2 Interactive Mask Painter

Run this cell to display the photo with a paintable canvas overlay. Draw over the damaged area, then click **Save Mask ✓**. The mask is sent back to Python and stored in `painted_mask`.

In [ ]:
import base64, io
from PIL import Image as PILImage
from IPython.display import display, HTML
from google.colab import output

IMAGE_PATH   = '/content/drive/MyDrive/pde_v2/content/test_image_8.PNG'
original_rgb = load_image(IMAGE_PATH, grayscale=False)

if len(original_rgb.shape) == 2:
    original_rgb = np.stack((original_rgb,)*3, axis=-1)

H3, W3, _ = original_rgb.shape

img_uint8 = (np.clip(original_rgb, 0, 1) * 255).astype(np.uint8)
pil_img   = PILImage.fromarray(img_uint8)
buf       = io.BytesIO()
pil_img.save(buf, format='PNG')
img_b64   = base64.b64encode(buf.getvalue()).decode('utf-8')

painted_mask = np.zeros((H3, W3), dtype=bool)

def save_mask(mask_b64):
    global painted_mask
    mask_bytes   = base64.b64decode(mask_b64)
    mask_img     = PILImage.open(io.BytesIO(mask_bytes)).convert('L')
    painted_mask = np.array(mask_img) > 128
    print(f'Mask saved: {painted_mask.sum()} pixels ({100*painted_mask.mean():.2f}%)')

output.register_callback('save_mask', save_mask)

DISPLAY_W = min(W3, 700)
DISPLAY_H = int(H3 * DISPLAY_W / W3)

html = f"""
<div style="font-family: monospace; user-select: none;">
  <div style="margin-bottom:8px; display:flex; gap:12px; align-items:center;">
    <label>Brush size:
      <input id="brushRange" type="range" min="2" max="60" value="12"
             oninput="document.getElementById('brushVal').textContent=this.value">
      <span id="brushVal">12</span>px
    </label>
    <button onclick="clearMask()" style="padding:4px 12px; cursor:pointer;">Clear</button>
    <button onclick="saveMask()"
      style="padding:4px 14px; background:#2a7; color:#fff; border:none; border-radius:4px; cursor:pointer; font-size:14px;">
      Save Mask ✓
    </button>
    <span id="status" style="color:#888;"></span>
  </div>
  <div style="position:relative; width:{DISPLAY_W}px; height:{DISPLAY_H}px;">
    <canvas id="imgCanvas"  width="{DISPLAY_W}" height="{DISPLAY_H}" style="position:absolute; top:0; left:0;"></canvas>
    <canvas id="drawCanvas" width="{DISPLAY_W}" height="{DISPLAY_H}" style="position:absolute; top:0; left:0; opacity:0.45; cursor:crosshair;"></canvas>
  </div>
</div>
<script>
const IMG_W = {W3}, IMG_H = {H3}, DISP_W = {DISPLAY_W}, DISP_H = {DISPLAY_H};
const imgCanvas  = document.getElementById('imgCanvas');
const imgCtx     = imgCanvas.getContext('2d');
const drawCanvas = document.getElementById('drawCanvas');
const drawCtx    = drawCanvas.getContext('2d');
drawCtx.fillStyle = 'red';

const photo = new Image();
photo.onload = () => imgCtx.drawImage(photo, 0, 0, DISP_W, DISP_H);
photo.src = 'data:image/png;base64,{img_b64}';

let painting = false;
const getBrush = () => parseInt(document.getElementById('brushRange').value);
function getPos(e) {{
  const rect = drawCanvas.getBoundingClientRect();
  const cx = e.touches ? e.touches[0].clientX : e.clientX;
  const cy = e.touches ? e.touches[0].clientY : e.clientY;
  return {{ x: cx - rect.left, y: cy - rect.top }};
}}
function paint(e) {{
  if (!painting) return;
  e.preventDefault();
  const {{x, y}} = getPos(e);
  drawCtx.beginPath(); drawCtx.arc(x, y, getBrush(), 0, Math.PI * 2); drawCtx.fill();
}}
drawCanvas.addEventListener('mousedown',  e => {{ painting = true; paint(e); }});
drawCanvas.addEventListener('mousemove',  paint);
drawCanvas.addEventListener('mouseup',    () => painting = false);
drawCanvas.addEventListener('mouseleave', () => painting = false);
function clearMask() {{
  drawCtx.clearRect(0, 0, DISP_W, DISP_H);
  document.getElementById('status').textContent = 'Cleared.';
}}
function saveMask() {{
  const fullCanvas = document.createElement('canvas');
  fullCanvas.width = IMG_W; fullCanvas.height = IMG_H;
  const fullCtx = fullCanvas.getContext('2d');
  fullCtx.drawImage(drawCanvas, 0, 0, IMG_W, IMG_H);
  const imageData = fullCtx.getImageData(0, 0, IMG_W, IMG_H);
  const grayCanvas = document.createElement('canvas');
  grayCanvas.width = IMG_W; grayCanvas.height = IMG_H;
  const grayCtx  = grayCanvas.getContext('2d');
  const grayData = grayCtx.createImageData(IMG_W, IMG_H);
  for (let i = 0; i < imageData.data.length; i += 4) {{
    const val = imageData.data[i + 3] > 10 ? 255 : 0;
    grayData.data[i] = grayData.data[i+1] = grayData.data[i+2] = val;
    grayData.data[i+3] = 255;
  }}
  grayCtx.putImageData(grayData, 0, 0);
  const maskB64 = grayCanvas.toDataURL('image/png').split(',')[1];
  document.getElementById('status').textContent = 'Sending...';
  google.colab.kernel.invokeFunction('save_mask', [maskB64], {{}});
  document.getElementById('status').textContent = 'Saved ✓';
}}
</script>
"""
display(HTML(html))
print(f'Image: {H3}×{W3}  — Paint over damage, then click "Save Mask ✓" before running the next cell.')

### 9.3 Run All Models on Painted Mask

In [ ]:
if 'painted_mask' not in globals() or painted_mask.sum() == 0:
    print('Mask is empty — paint in the cell above and click Save Mask first.')
else:
    print(f"Running all 4 models on {IMAGE_PATH}...")
    print(f"Mask: {painted_mask.sum()} px ({100*painted_mask.mean():.2f}%)")

    print("\nRunning Harmonic...")
    restored_harmonic = np.clip(np.stack([
        harmonic_inpaint(original_rgb[:,:,c], painted_mask, max_iter=5000, verbose=False)[0]
        for c in range(3)], axis=-1), 0, 1)

    print("Running TV...")
    restored_tv = np.clip(np.stack([
        tv_inpaint(original_rgb[:,:,c], painted_mask, max_iter=8000, verbose=False)[0]
        for c in range(3)], axis=-1), 0, 1)

    print("Running CDD...")
    restored_cdd, _ = universal_cdd_inpaint(original_rgb, painted_mask, mode='photo', verbose=True)
    restored_cdd    = np.clip(restored_cdd, 0, 1)

    print("Running QCDD...")
    restored_qcdd, _ = universal_qcdd_inpaint(original_rgb, painted_mask, mode='photo', verbose=True)
    restored_qcdd    = np.clip(restored_qcdd, 0, 1)

    mask_overlay = original_rgb.copy()
    mask_overlay[painted_mask] = [1, 0, 0]

    items = [
        (original_rgb,      'Original Photo', 'manual_original.pdf'),
        (mask_overlay,      'Painted Mask',   'manual_masked.pdf'),
        (restored_harmonic, 'Harmonic',        'manual_harmonic.pdf'),
        (restored_tv,       'TV',              'manual_tv.pdf'),
        (restored_cdd,      'CDD',             'manual_cdd.pdf'),
        (restored_qcdd,     'QCDD',            'manual_qcdd.pdf'),
    ]
    for img, title, fname in items:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(img); ax.set_title(title, fontsize=23, fontweight='bold', pad=8); ax.axis('off')
        plt.tight_layout()
        plt.savefig(fname, bbox_inches='tight', dpi=150)
        plt.show(); print(f"Saved: {fname}")

    PILImage.fromarray((restored_cdd * 255).astype('uint8'), mode='RGB').save('/content/restored_cdd.jpg')
    print('Saved: /content/restored_cdd.jpg')